In [11]:
!nvidia-smi

Tue Nov 24 03:55:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [13]:

%%writefile matrix.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 4

__global__ void multiply(int *A,int *B,int *C)
{
    int row=threadIdx.y+blockIdx.y*blockDim.y;
    int col=threadIdx.x+blockIdx.x*blockDim.x;

    if(row<N && col<N)
    {
        int sum=0;
        for(int k=0;k<N;k++)
            sum+=A[row*N+k]*B[k*N+col];
        C[row*N+col]=sum;
    }
}

int main()
{
    int A[N][N]={{1,2,3,4},{5,6,7,8},
                 {9,10,11,12},{13,14,15,16}};

    int B[N][N]={{1,2,3,4},{5,6,7,8},
                 {9,10,11,12},{13,14,15,16}};

    int C_CPU[N][N],C_GPU[N][N];

    // CPU Matrix Multiplication
    clock_t cpu_start=clock();

    for(int i=0;i<N;i++)
    {
        for(int j=0;j<N;j++)
        {
            C_CPU[i][j]=0;
            for(int k=0;k<N;k++)
                C_CPU[i][j]+=A[i][k]*B[k][j];
        }
    }

    clock_t cpu_end=clock();

    double cpu_time=(double)(cpu_end-cpu_start)
                    /CLOCKS_PER_SEC*1000.0;

    // GPU Memory Allocation
    int *d_A,*d_B,*d_C;

    cudaMalloc(&d_A,sizeof(A));
    cudaMalloc(&d_B,sizeof(B));
    cudaMalloc(&d_C,sizeof(C_GPU));

    cudaMemcpy(d_A,A,sizeof(A),cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,sizeof(B),cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks(1,1);

    // GPU Timing
    cudaEvent_t start,stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warm-up
    multiply<<<blocks,threads>>>(d_A,d_B,d_C);
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    multiply<<<blocks,threads>>>(d_A,d_B,d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time=0;
    cudaEventElapsedTime(&gpu_time,start,stop);

    cudaMemcpy(C_GPU,d_C,sizeof(C_GPU),cudaMemcpyDeviceToHost);

    // Display Result
    printf("Result Matrix (GPU):\n");

    for(int i=0;i<N;i++)
    {
        for(int j=0;j<N;j++)
            printf("%d ",C_GPU[i][j]);
        printf("\n");
    }

    // Verification
    int correct=1;

    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++)
            if(C_CPU[i][j]!=C_GPU[i][j])
                correct=0;

    printf("\nCPU Execution Time: %.6f ms\n",cpu_time);
    printf("GPU Kernel Execution Time: %.6f ms\n",gpu_time);
    printf("Result Verification: %s\n",correct?"PASSED":"FAILED");

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}


Overwriting matrix.cu


In [14]:
!nvcc matrix.cu -o matrix

In [15]:
!./matrix

Result Matrix (GPU):
90 100 110 120 
202 228 254 280 
314 356 398 440 
426 484 542 600 

CPU Execution Time: 0.002000 ms
GPU Kernel Execution Time: 0.018432 ms
Result Verification: PASSED
